# Dataset Inspection: Finding the Problems Before They Find You

**Estimated time:** 2–3 hours (including exercises and capstone)

In the previous lesson you analyzed a clean dataset. Real datasets are **never** clean. This lesson teaches the skill that separates careful analysts from careless ones: **systematically inspecting a raw dataset to uncover its problems *before* trusting any result computed from it.**

## Why this matters more than any other skill

Most data problems do **not** produce error messages. Pandas will happily:

- compute an average that includes placeholder values like `99`,
- split one branch into three groups because of a stray space,
- double-count revenue because of duplicated rows,
- "multiply" text and produce garbage without complaining.

The result *looks* like a number. It gets put in a report. Decisions get made. This failure mode has a name: **garbage in, gospel out** — wrong inputs producing outputs that are trusted because they came from code.

## What you'll learn

1. The typical issues found in raw datasets, and exactly *how* each one corrupts an analysis
2. **Manual** inspection: reading the raw file before pandas touches it
3. The **pandas inspection toolkit**: what each method reports and how to interpret its output
4. How to quantify problems (counts and percentages, not vibes)
5. A reusable **inspection protocol** you can apply to any dataset
6. Capstone: produce a full *data quality report* for an unseen dirty dataset

**Scope note:** this lesson is about **finding and diagnosing** issues. *Fixing* them (data cleaning) is its own craft and comes in a later lesson — though section 12 previews which tool fixes what.

## Prerequisites

The previous lesson ("Introduction to Data Analysis"): loading CSVs, selecting/filtering, `groupby`, basic plotting.

---

## 2. Setup: our crime scene 🕵️

The coffee shop chain from last lesson has a new point-of-sale system, and you've been handed its **raw export** for March: `coffee_sales_raw.csv`. Head office already computed some numbers from it. Your job today: find out whether this file can be trusted.

The cell below generates the raw export (plus a second file for the capstone). As before, **you don't need to study the generation code** — in real life these files would simply land in your inbox. Just run it.

In [1]:
# Run this cell to create the raw data files. You don't need to study this code.
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)
n = 400

products = {
    "Espresso": ("Coffee", 2.50), "Latte": ("Coffee", 4.00),
    "Cappuccino": ("Coffee", 3.80), "Green Tea": ("Tea", 3.00),
    "Chai Latte": ("Tea", 4.20), "Croissant": ("Food", 3.50),
    "Sandwich": ("Food", 6.50), "Muffin": ("Food", 3.20),
}
names = list(products.keys())
chosen = rng.choice(names, size=n, p=[0.18, 0.22, 0.15, 0.08, 0.07, 0.12, 0.08, 0.10])

df0 = pd.DataFrame({
    "date": rng.choice(pd.date_range("2026-03-01", "2026-03-31"), size=n),
    "branch": rng.choice(["Downtown", "Airport", "University"], size=n, p=[0.45, 0.25, 0.30]),
    "product": chosen,
    "category": [products[p][0] for p in chosen],
    "quantity": rng.integers(1, 5, size=n),
    "unit_price": [f"{products[p][1]:.2f}" for p in chosen],
    "customer_rating": np.round(rng.uniform(2.5, 5.0, size=n), 1),
}).sort_values("date").reset_index(drop=True)
df0["date"] = df0["date"].dt.strftime("%Y-%m-%d")

def pick(k, exclude=()):
    pool = [i for i in range(n) if i not in exclude]
    return rng.choice(pool, size=k, replace=False)

# --- inject issues (deterministic thanks to the seed) ---
used = set()
i = pick(40); df0.loc[i, "customer_rating"] = np.nan; used |= set(i)          # real missing
i = pick(10, used); df0.loc[i, "customer_rating"] = 99.0                       # sentinel
i = pick(25); df0.loc[i, "unit_price"] = "$" + df0.loc[i, "unit_price"]        # $ prefix
i = pick(10, set(i)); df0.loc[i, "unit_price"] = "-"                           # disguised missing
i = pick(30); df0.loc[i, "branch"] = df0.loc[i, "branch"].str.lower()          # casing
i = pick(20, set(i)); df0.loc[i, "branch"] = df0.loc[i, "branch"] + " "        # trailing space
i = pick(8, set(i)); df0.loc[i, "branch"] = df0.loc[i, "branch"].str.upper()   # CAPS
i = pick(4); df0.loc[i, "category"] = "Cofee"                                  # misspelling
i = pick(5); df0.loc[i, "quantity"] = -df0.loc[i, "quantity"]                  # impossible
i = pick(6); df0.loc[i, "date"] = pd.to_datetime(df0.loc[i, "date"]).dt.strftime("%d/%m/%Y")
dupes = df0.iloc[rng.choice(n, size=8, replace=False)]                         # exact duplicates
df0 = pd.concat([df0, dupes]).sort_values("date").reset_index(drop=True)
df0.to_csv("coffee_sales_raw.csv", index=False)

# --- capstone file: supplier deliveries (issues listed in the solutions) ---
m = 120
sup_base = rng.choice(["BeanCo", "MilkFarm", "BakeHouse"], size=m, p=[0.4, 0.3, 0.3])
items = {"BeanCo": "Arabica beans", "MilkFarm": "Whole milk", "BakeHouse": "Pastry dough"}
cap = pd.DataFrame({
    "delivery_id": [f"D{1000+j}" for j in range(m)],
    "supplier": sup_base,
    "item": [items[s] for s in sup_base],
    "quantity_kg": np.round(rng.uniform(5, 80, size=m), 1).astype(str),
    "cost": [f"{v:,.2f}" for v in rng.uniform(200, 3000, size=m)],
    "delivery_date": pd.Series(rng.choice(pd.date_range("2026-03-01", "2026-03-31"), size=m)
                               ).dt.strftime("%Y-%m-%d"),
})
i = rng.choice(m, 12, replace=False); cap.loc[i, "supplier"] = cap.loc[i, "supplier"].str.lower()
i = rng.choice(m, 8, replace=False);  cap.loc[i, "supplier"] = cap.loc[i, "supplier"] + " "
i = rng.choice(m, 3, replace=False);  cap.loc[i, "item"] = "Arabca beans"
i = rng.choice(m, 7, replace=False);  cap.loc[i, "quantity_kg"] = "unknown"
i = rng.choice(m, 4, replace=False);  cap.loc[i, "quantity_kg"] = "-12.0"
cap.loc[rng.choice(m, 1), "quantity_kg"] = "9999.0"
i = rng.choice(m, 20, replace=False); cap.loc[i, "cost"] = "$" + cap.loc[i, "cost"]
i = rng.choice(m, 5, replace=False)
cap.loc[i, "delivery_date"] = pd.to_datetime(cap.loc[i, "delivery_date"]).dt.strftime("%d/%m/%Y")
cap = pd.concat([cap, cap.iloc[rng.choice(m, 5, replace=False)]]).reset_index(drop=True)
cap.columns = ["delivery_id", "supplier ", "item", "quantity_kg", " cost", "delivery_date"]
with open("supplier_deliveries_raw.csv", "w") as f:
    f.write("Supplier delivery export - INTERNAL USE ONLY\n")
    f.write("Generated by LogiTrack v2.3 on 2026-04-01\n")
    f.write(cap.to_csv(index=False))

print("Created coffee_sales_raw.csv and supplier_deliveries_raw.csv")

Created coffee_sales_raw.csv and supplier_deliveries_raw.csv


---

## 3. The trap: what happens if you *don't* inspect

Let's play the careless analyst first. Load the file, compute the numbers management asked for, done in five minutes:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("coffee_sales_raw.csv")
df.head(3)  # looks fine at a glance...

In [ ]:
# "How many sales per branch?"
df["branch"].value_counts()

**Eleven branches?!** The chain has three. `Downtown`, `downtown` and `Downtown ` (note the trailing space — you can't even see it) are counted as different branches. Any per-branch number built on this is wrong.

In [ ]:
# "What's our average customer rating?"
print(df["customer_rating"].mean().round(2), "out of 5")

An average of ~6.3 **on a 1-to-5 scale**. Something impossible is hiding in that column, and `.mean()` didn't warn us. (Notice this one at least *smells* wrong. The truly dangerous cases are when the corrupted value looks plausible — imagine the sentinel had been `4.9` instead of `99`.)

In [ ]:
# "What's the total revenue?" -- quantity times price, like last lesson
revenue = df["quantity"] * df["unit_price"]
revenue.head(8)

No error — but look closely at the output. `unit_price` was loaded as **text**, and multiplying text by an integer in Python *repeats the text*: `"6.50" * 2` gives `"6.506.50"`. Pandas computed 400+ rows of nonsense without a single complaint. If any step later coaxes these into numbers, `6.506.5` becomes a price.

> **The lesson:** none of these three disasters raised an error. The only defense is inspection *before* analysis. Let's learn to do it properly.

---

## 4. Manual inspection: look at the raw file first

Before pandas ever touches a file, look at the file itself. `read_csv` makes dozens of silent decisions (delimiter, header row, types, quoting) — if the file violates its assumptions, the damage happens **at load time** and everything downstream inherits it.

### 4.1 Reading raw lines with Python

You can open a CSV in any text editor (Notepad, VS Code) — or peek at it right here. Using `repr()` makes invisible characters visible:

In [ ]:
with open("coffee_sales_raw.csv") as f:
    for line in [next(f) for _ in range(8)]:
        print(repr(line))

How to read this, line by line:

- **Line 1 is the header.** Good: the first line contains column names, comma-separated, no junk above it. If a file starts with metadata lines (export banners, timestamps), `read_csv` will mangle everything unless you skip them — you'll meet this in the capstone.
- **The delimiter is a comma.** Files from European locales often use `;` (because `,` is their decimal separator). Loading a `;`-file with default settings yields a DataFrame with **one giant column** — instantly visible if you look.
- **`repr()` exposes hidden characters**: the `\n` line endings, and — if you scan carefully — values like `'DOWNTOWN '` with a trailing space, and prices like `'$3.50'`. These are exactly the landmines from section 3.
- **Quoting**: values containing commas should be wrapped in quotes (`"Smith, John"`). Unquoted embedded commas shift every subsequent value one column to the right — a catastrophic, hard-to-spot corruption.

### 4.2 What about opening it in Excel?

Spreadsheets are fine for a quick look, but know their traps: Excel **silently converts** things on open — long numbers lose precision, `03-04` becomes a date, leading zeros vanish (`0721` → `721`). Never re-save a raw file from Excel; inspect a *copy* if you must.

### 4.3 The limits of manual inspection

Your eyes are excellent at spotting *structural* problems (junk headers, delimiters, quoting) in the first dozen lines — and hopeless at scanning 400 rows for a stray space, let alone 4 million. Manual inspection answers "*can pandas load this file sanely?*". Everything after that needs programmatic tools.

### ✏️ Exercises 4

**4.1** Print the **last** 5 raw lines of the file (hint: `f.readlines()` gives a list — slice it). Files often accumulate junk at the *end* too: totals rows, footers, half-written records.

**4.2** In the raw lines you've printed, find at least one value with a `$` and one branch name with wrong casing. (Just look — no code needed beyond printing.)

In [ ]:
# Your solutions for exercises 4 here


---

## 5. The structured first look: `shape`, `head`, `tail`, `sample`

Now that we know the file *loads* sanely, we inspect the loaded DataFrame. Always start with size and a few actual rows.

In [ ]:
df.shape

**Interpretation:** always check the row count against expectations. The shop makes roughly 13 sales/day × 31 days ≈ 400 — but we have 408. A count that's *too high* suggests duplicates or leftover junk rows; *too low* suggests a truncated export. Row count is your first plausibility test, and it requires **domain knowledge** — inspection is always a dialogue between the data and what you know about reality.

In [ ]:
df.head(3)

In [ ]:
df.tail(3)

`head()` and `tail()` show the edges of the file — useful because exports often have problems exactly there (headers at the top, totals/footers at the bottom). But the edges are **not representative**: files are usually sorted, so the first rows might all be one date or one branch, and issues that live in the middle stay invisible. That's what `sample()` is for:

In [ ]:
df.sample(8, random_state=1)  # 8 random rows; random_state makes it repeatable

**Interpretation:** in a random sample you're likely to stumble on the oddities — scan it for anything that looks off (a `$`, a `-`, a lowercase branch, a negative number). Make sampling a habit; run it a few times without `random_state` to see different rows.

### ✏️ Exercises 5

**5.1** Using code, compute how many rows more than the "expected" 400 this file has.

**5.2** Draw three different samples of 10 rows. List every suspicious thing you spot.

In [ ]:
# Your solutions for exercises 5 here


---

## 6. `df.info()`: the single most informative line of inspection code

If you run only one method on a new dataset, run this one.

In [ ]:
df.info()

Let's interpret **every part** of this output:

**1. `RangeIndex: 408 entries`** — the row count again (408, not 400 — noted).

**2. The `Non-Null Count` column** — for each column, how many values are *not* missing. Read it as a subtraction: any column whose count is **below 408** contains missing values. Here `customer_rating` shows 367 non-null → 41 explicitly missing values. But **beware the reverse**: a full non-null count does *not* mean "no missing data". `unit_price` shows 408 non-null, yet we know some values are the *text* `"-"` — pandas doesn't recognize that as missing. `info()` only counts *explicit* `NaN`. Disguised missing values are invisible here; we hunt them in section 7.

**3. The `Dtype` column** — compare each dtype against what the column *should* be:

| Column | Expected | Actual | Verdict |
|---|---|---|---|
| `date` | `datetime64` | `object`/`str` | 🚩 dates loaded as text |
| `branch`, `product`, `category` | text | text | ✅ |
| `quantity` | `int64` | `int64` | ✅ (but dtype can't reveal *impossible values* — see §10) |
| `unit_price` | `float64` | `object`/`str` | 🚩🚩 **a number column loaded as text** |
| `customer_rating` | `float64` | `float64` | ✅ dtype-wise |

The `unit_price` flag is the big one. **Whenever a column that should be numeric has dtype `object`/`str`, something inside it is not a number** — pandas only falls back to text when at least one value forces it to. The dtype doesn't tell you *which* values are guilty; section 8 shows how to find them.

💡 A subtlety worth memorizing: **one single bad value poisons the whole column's dtype.** 407 clean prices + one `"N/A"` = a text column. This is why `info()` is such a sensitive detector.

**4. `memory usage`** — rarely an inspection concern at this size; becomes relevant for huge files.

### ✏️ Exercises 6

**6.1** From the `info()` output alone: exactly how many values are missing in `customer_rating`? Verify with `df["customer_rating"].isna().sum()`.

**6.2** Explain in one sentence why `date` having dtype `object`/`str` would break the analysis question *"show me a line chart of daily revenue"*.

In [ ]:
# Your solutions for exercises 6 here


---

## 7. `df.describe()`: statistics as a lie detector

`describe()` computes summary statistics — but during inspection, we read it not to *learn about the data* but to check **plausibility**.

In [ ]:
df.describe()

How to interrogate each row of this output (numeric columns only by default):

- **`count`** — number of non-missing values; cross-checks `info()`.
- **`min` / `max` — the plausibility bounds check.** This is the most valuable inspection read:
  - `quantity` min is **negative**. A negative number of coffees sold is not an outlier, it's an **impossibility** — either a data-entry error or a meaning we don't understand (some systems encode refunds as negative sales — which would change the analysis entirely! Investigate before assuming.)
  - `customer_rating` max is **99** on a 1–5 scale. That's a **sentinel value**: a placeholder some systems write to mean "no rating given". It's a *disguised missing value wearing a numeric costume* — the most dangerous species, because every statistic silently digests it.
- **`mean` vs `50%` (median) — the skew/contamination signal.** For an honest, roughly symmetric column these are close. `customer_rating` shows mean ≈ 6.3 but median ≈ 3.8 — a large gap means a few extreme values are dragging the mean. The median resists them; the mean doesn't. **A mean far from its median is always worth investigating.**
- **`std`** — an absurdly large standard deviation (compare it to the mean) is another extreme-value alarm.
- **`25%` / `75%`** — the middle half of the data. If min/max are far outside this range, the extremes are rare — consistent with a few bad values rather than a genuinely wild column.

### `describe(include="all")`: extending the lie detector to text columns

In [ ]:
df.describe(include="all")

For text columns, three new rows appear:

- **`unique`** — the number of distinct values. **Check it against domain knowledge:** `branch` should have 3 unique values; it has 11 → categorical inconsistency (casing/whitespace variants). `category` should have 3; it has 4 → something's misspelled. `unique` is the fastest inconsistency alarm there is.
- **`top` / `freq`** — the most common value and its count. Useful sanity check: if `top` for a "customer name" column were `"N/A"` with freq 5000, you'd have found a disguised-missing epidemic in one glance.

### ✏️ Exercises 7

**7.1** From `describe(include="all")`: how many distinct values does `date` have, and why is that number itself a small red flag if you expected "one row per sale, March only"? (Hint: how many days are in March? What would *more* distinct values than that mean, given the dtype?)

**7.2** Compute mean and median of `quantity`. Are they close? What does that suggest about how contaminated this column is, compared to `customer_rating`?

In [ ]:
# Your solutions for exercises 7 here


---

## 8. Missing values: the explicit and the disguised

### 8.1 Explicit missing values (`NaN`)

These are the honest ones — pandas knows they're missing.

In [ ]:
df.isna().sum()

In [ ]:
# Counts are good; PERCENTAGES are better for judging severity
(df.isna().mean() * 100).round(1)

**Interpretation:** ~10% of ratings are missing. Whether that's a problem depends on the question:

- For *"average rating per product"* it might be fine — **if** the missingness is random.
- But ask **why** values are missing. If unhappy customers skip the rating (very plausible!), the surviving ratings are biased upward, and no amount of statistics on the remaining values fixes that. *The pattern of missingness matters as much as the amount.* A quick check: does missingness concentrate somewhere?

In [ ]:
# Are missing ratings spread evenly across branches, or concentrated?
df.groupby(df["branch"].str.strip().str.lower())["customer_rating"] \
  .apply(lambda s: s.isna().mean() * 100).round(1)

(We normalize the branch labels on the fly here purely for a fair comparison — full cleaning still comes later. Roughly even percentages → missingness doesn't obviously depend on branch. Good news, as far as it goes.)

### 8.2 Disguised missing values

Now the dangerous ones: values that *mean* "missing" but *look* like data. Common disguises: `"N/A"`, `"NA"`, `"-"`, `"?"`, `"none"`, `"unknown"`, the empty string `""`, and numeric sentinels like `0`, `-1`, `99`, `999`, `-9999`.

**Detection tool #1: `value_counts(dropna=False)`** — the full census of a column. For text columns, disguises appear as ordinary-looking entries:

In [ ]:
df["unit_price"].value_counts(dropna=False)

**Interpretation:** the census exposes two impostors at once — a `-` entry (disguised missing) and `$`-prefixed twins of ordinary prices (`3.50` *and* `$3.50` living as separate values). Also note `dropna=False`: without it, `value_counts` hides real `NaN`s, and during inspection you want to see *everything*.

💡 **A useful quirk:** `read_csv` already auto-recognizes *some* disguises — tokens like `"N/A"`, `"NA"`, `"NULL"`, `"nan"` are converted to real `NaN` at load time (see its `na_values` parameter). That's why the disguises that *survive* into your DataFrame are precisely the nonstandard ones — `"-"`, `"?"`, `"unknown"`, `"missing"` — and why you can never assume the load caught everything.

**Detection tool #2: the sentinel hunt in numeric columns.** Sentinels like `99` don't show up in `isna()` — they show up as *impossible values* (§7's max check) or as an unnatural **spike in the distribution**:

In [ ]:
df["customer_rating"].plot(kind="hist", bins=30,
                              title="customer_rating -- see the lonely spike at 99")
plt.xlabel("rating")
plt.show()

In [ ]:
# Quantify the spike
(df["customer_rating"] == 99).sum()

**Interpretation:** a healthy rating distribution lives between 1 and 5; an isolated island at exactly 99, hit repeatedly, is no coincidence — real measurements don't pile up on one absurd value. Ten sales have "rating 99", i.e., ten *more* missing ratings wearing a costume. Combined with the 41 explicit `NaN`s, the true missing rate is ~12.5%, not 10%.

⚠️ **Why sentinels are the most dangerous issue in this entire lesson:** they pass every type check, every `isna()` check, and quietly shift every average, sum, and correlation. The only defenses are range checks against domain knowledge and distribution plots.

### ✏️ Exercises 8

**8.1** Run `value_counts(dropna=False)` on `category`. Two findings hide in it — one about spelling, one about what *isn't* there (are there any missing values in this column?).

**8.2** How many rows have `unit_price` equal to the string `"-"`? What *percentage* of the dataset is that?

**8.3** A colleague proposes: "0 quantity probably means missing, let's check." Test whether `quantity` contains any zeros. (It's a good instinct — always test the hypothesis rather than assume.)

In [ ]:
# Your solutions for exercises 8 here


---

## 9. Type problems: finding exactly which values are guilty

`info()` told us `unit_price` and `date` have the wrong dtype. That's the *symptom*. Inspection means locating the *cause*: which specific values prevent the proper type?

### 9.1 Why wrong types corrupt silently

A tiny demonstration of the two failure modes — sorting and arithmetic on "numbers" that are text:

In [ ]:
as_text = pd.Series(["9", "10", "2"])
print("Text sort:   ", sorted(as_text))          # '10' comes before '9'!
print("Numeric sort:", sorted(as_text.astype(int)))
print("Text 'math': ", pd.Series(["4.00"]) [0] * 2)   # repetition, not multiplication

Text sorts **character by character**, so `"10" < "9"` (because the character `"1"` < `"9"`). Rankings, min/max, and "top 10" lists computed on text-numbers are quietly wrong.

### 9.2 The coercion diagnostic: `pd.to_numeric(errors="coerce")`

The single most useful type-inspection pattern. `errors="coerce"` converts what it can and turns everything unconvertible into `NaN` — and those *new* NaNs are precisely a list of the guilty values:

In [ ]:
converted = pd.to_numeric(df["unit_price"], errors="coerce")
guilty = df["unit_price"].notna() & converted.isna()   # was a value, refused to convert
print("Unconvertible values:", guilty.sum())
df.loc[guilty, "unit_price"].value_counts()

**Interpretation:** the diagnosis is now precise — the column fails to be numeric for **two distinct reasons**: `$`-prefixed prices (a formatting artifact, the numbers underneath are real) and `-` strings (disguised missing — no number exists). This distinction matters enormously for the *future* cleaning step: `$3.50` should become `3.50`, while `-` should become `NaN`. **Good inspection doesn't just find problems; it classifies them.**

💡 Other frequent numeric saboteurs this pattern catches: thousands separators (`"1,200"`), units in the value (`"3.5kg"`), spaces (`" 4.00"`), and locale decimals (`"4,00"`).

### 9.3 The same trick for dates: `pd.to_datetime`

We *expect* ISO dates (`2026-03-05`). Test the expectation explicitly:

In [ ]:
parsed = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
print("Rows not matching the expected format:", parsed.isna().sum())
df.loc[parsed.isna(), "date"].unique()

**Interpretation:** a handful of rows use `DD/MM/YYYY` instead. Mixed date formats are among the nastiest issues in raw data, because a value like `05/03/2026` is **ambiguous** — March 5th or May 3rd? The digits alone cannot tell you; you must consult the data's source (which country? which software?). An inspection report should flag ambiguity explicitly rather than guess.

⚠️ Note the method: we tested against the format we **expected**, rather than letting pandas guess. Inspection is hypothesis testing — "I believe this column is X" → verify → the exceptions are your findings.

### ✏️ Exercises 9

**9.1** Apply the coercion diagnostic to `quantity`. Does it find any non-numeric values? (A clean result is also a finding — you've *verified* the column is numerically sound, even though §10 will show numeric ≠ correct.)

**9.2** Predict, then check: what does `df["date"].min()` return, and why is the *string* minimum of a date column with mixed formats untrustworthy? (Think about how `"05/03/2026"` compares to `"2026-03-01"` as text.)

In [ ]:
# Your solutions for exercises 9 here


---

## 10. Categorical inconsistency: one thing, many names

The chain has 3 branches; `describe()` claimed 11. Time to see exactly what's going on. The workhorse is again the full census:

In [ ]:
df["branch"].value_counts(dropna=False)

**Interpretation — read a census like this in three passes:**

1. **Cluster the variants.** `Downtown`, `downtown`, `DOWNTOWN`, `Downtown ` are one real-world branch fractured into four labels. Fracture sources: hand-typed input, different software versions, copy-paste with trailing spaces.
2. **Check the tail.** Low-frequency values at the bottom of a census are *suspect by position*: genuine categories usually appear often; rare strays are typos or corruption.
3. **Mind the invisible.** `Downtown ` vs `Downtown` — can you even see the difference? Trailing whitespace is invisible in most displays. Prove it exists with code, never with eyes:

In [ ]:
has_hidden_space = (df["branch"] != df["branch"].str.strip()).sum()
print("Values with leading/trailing whitespace:", has_hidden_space)

In [ ]:
# How many DISTINCT branches are there really, once case and spaces are ignored?
print("Raw distinct values:       ", df["branch"].nunique())
print("After strip + lowercase:   ", df["branch"].str.strip().str.lower().nunique())

**The two-number comparison above is a beautiful one-liner verdict:** 11 raw labels collapse to 3 normalized ones → every extra label was a casing/whitespace artifact, and the true structure (3 branches) matches domain knowledge. If normalization had yielded, say, 4, you'd know a *genuinely different* label hides in there (a misspelling, a closed branch, a test entry) — which is exactly the situation in `category`:

In [ ]:
print(df["category"].value_counts())
print()
print("After normalization:", df["category"].str.strip().str.lower().nunique(), "distinct values")

**Interpretation:** normalization does *not* rescue `category` — 4 stays 4, because `Cofee` isn't a casing variant of `Coffee`, it's a **misspelling** (4 occurrences, suspiciously rare → tail rule). Distinguishing *normalizable* variants from *genuine* strays is the key skill of categorical inspection.

**Why this issue class is so destructive:** every `groupby`, every filter (`df[df["branch"] == "Downtown"]` silently *drops* the misspelled rows), every count and join splits or loses data — with zero errors.

### ✏️ Exercises 10

**10.1** Run the full census on `product`. Is this column consistent, or does it have variants/misspellings? Support your verdict with the raw-vs-normalized `nunique` comparison.

**10.2** How many rows would the filter `df[df["branch"] == "Downtown"]` silently *miss* (i.e., rows that are really Downtown but labeled differently)? Compute it.

In [ ]:
# Your solutions for exercises 10 here


---

## 11. Duplicates: counting things twice

`shape` said 408 rows against an expected ~400. Prime suspect: duplicated rows — the same sale exported twice (a classic system glitch: retries, overlapping exports, merged files).

### Detecting exact duplicates

In [ ]:
df.duplicated().sum()

`duplicated()` marks a row `True` if an **identical row appeared earlier** — so 9 means "9 rows are repeat offenders" (the first occurrence of each is not counted). To *look* at the duplicate families, use `keep=False`, which marks **all** members:

In [ ]:
df[df.duplicated(keep=False)].sort_values(["date", "branch", "product"]).head(6)

**Interpretation — the crucial question: are they *true* duplicates?** Two identical rows can mean:

- **The same event recorded twice** → genuine duplication, inflates every total. Likely here, since a sale export should be one row per transaction.
- **Two genuinely identical events** → *not* an error! (Curious detail: `shape` showed only 8 rows more than expected, yet `duplicated()` found 9 — meaning at least one identical pair was very likely *two real, coincidentally identical sales*. The two checks disagree slightly, and that disagreement is informative.) Two customers really can buy 2 lattes at Downtown on the same day. Whether identical rows are suspicious depends entirely on whether the columns *should* uniquely identify an event. Our export has no transaction-ID column — a data-quality finding in itself! With an ID column, `df.duplicated(subset=["transaction_id"])` would give a definitive answer.

**Quantify the damage** — how much would naive numbers be inflated?

In [ ]:
dupes = df.duplicated().sum()
print(f"{dupes} duplicate rows = {dupes / len(df) * 100:.1f}% of all rows")
print(f"Quantity total counted twice: {df[df.duplicated()]['quantity'].sum()} items")

### ✏️ Exercises 11

**11.1** Check for *partial* duplicates: how many rows share the same `date`, `branch`, `product` **and** `quantity` (hint: `duplicated(subset=[...])`)? Why is this count larger than the exact-duplicate count, and why is it much weaker evidence of an error?

**11.2** In one sentence: why is "just delete all duplicates" a decision for the *cleaning* phase that needs domain input, rather than an automatic inspection-time action?

In [ ]:
# Your solutions for exercises 11 here


---

## 12. Impossible values and outliers: two different beasts

The final issue class. Keep these concepts strictly apart:

| | **Impossible value** | **Outlier** |
|---|---|---|
| Definition | Violates a *rule of reality* | Extreme but *possible* |
| Example | quantity = −3, rating = 99, age = 250 | one order of 40 sandwiches |
| Detected by | **domain rules** (explicit checks) | **statistics** (distributions, quantiles) |
| Verdict | always wrong (or misunderstood encoding) | maybe real! investigate, never auto-delete |

### 12.1 Hunting impossible values: write your rules as code

For every column, ask: *what range does reality allow?* Then test it:

In [ ]:
rules = {
    "quantity must be positive":        (df["quantity"] <= 0),
    "rating must be between 1 and 5":   (df["customer_rating"].notna()
                                          & ~df["customer_rating"].between(1, 5)),
}
for rule, violations in rules.items():
    print(f"{rule:38s} -> {violations.sum()} violations")

In [ ]:
df.loc[df["quantity"] <= 0, ["date", "branch", "product", "quantity"]]

**Interpretation:** five sales with negative quantity. Before labeling them "errors", entertain the alternative: some point-of-sale systems record **refunds** as negative sales. If so, they're not dirt — they're *meaning*, and deleting them would corrupt the revenue analysis in the other direction. The inspection report should say: *"5 negative quantities found — clarify with the system owner whether these encode refunds."* **Inspection produces questions as often as verdicts.**

### 12.2 Hunting outliers: let the distribution speak

Statistical outliers need a picture. A **boxplot** draws the middle 50% as a box and flags points far outside as dots; a **histogram** shows the full shape:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
df.loc[df["quantity"] > 0, "quantity"].plot(kind="box", ax=axes[0],
        title="quantity (impossible values excluded)")
df.loc[df["customer_rating"] <= 5, "customer_rating"].plot(
    kind="hist", bins=15, ax=axes[1], title="customer_rating (sentinels excluded)")
plt.tight_layout()
plt.show()

**Interpretation:** once the impossible values are set aside, both distributions look healthy — no extreme dots, a plausible shape. That's the correct order of operations: **remove the impossible from view first**, *then* judge outliers; otherwise sentinels masquerade as outliers and drown the real signal.

A quantile check is the numeric companion to the plots — compare the far tails to the bulk:

In [ ]:
df["quantity"].quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 1.0])

**Reading quantiles:** if the 99th percentile is 4 but the max were 400, exactly the top sliver would be wild → a few suspect rows. Here the picture is the mirror image at the *bottom*: the negatives live entirely below the ~1st percentile, i.e., a tiny tail — consistent with a handful of entry errors (or refund records) rather than a systemic problem.

⚠️ **The golden rule of outliers:** an outlier is *information* until proven otherwise. The largest order in the dataset might be your best customer, not your worst data point. Inspection flags; humans decide.

### ✏️ Exercises 12

**12.1** Write a domain rule for `unit_price` (after coercing it with `pd.to_numeric(..., errors="coerce")`): prices must be between 0.50 and 20. How many violations?

**12.2** Which product has the single largest *absolute* quantity in one sale? Is it an impossible value, an outlier, or normal? Justify using the concepts from the table above.

In [ ]:
# Your solutions for exercises 12 here


---

## 13. The inspection protocol 📋

Everything above, condensed into the routine to run on **every** new dataset — in this order:

```python
# STEP 0 -- MANUAL: peek at the raw file (text editor or open())
#   header ok? delimiter? junk lines top/bottom? quoting? suspicious tokens?

# STEP 1 -- SIZE & SHAPE                        expectation check
df.shape                                        # row count plausible?

# STEP 2 -- EYEBALL REAL ROWS
df.head(); df.tail(); df.sample(10)             # edges AND random middle

# STEP 3 -- STRUCTURE
df.info()                                       # non-null counts, dtypes vs expected
df.columns                                      # stray spaces in names?

# STEP 4 -- STATISTICAL PLAUSIBILITY
df.describe(include="all")                      # min/max possible? mean~median?
                                                # unique counts vs domain knowledge?
# STEP 5 -- MISSING (explicit + disguised)
df.isna().mean() * 100                          # percentages per column
df["col"].value_counts(dropna=False)            # censuses: N/A, -, ?, "" ...
# histograms of numerics                        # sentinel spikes (0, 99, -1, 999)

# STEP 6 -- TYPES: locate the guilty values
pd.to_numeric(df["col"], errors="coerce")       # new NaNs = the culprits
pd.to_datetime(df["col"], format="...", errors="coerce")

# STEP 7 -- CATEGORICALS
df["col"].nunique()  vs  df["col"].str.strip().str.lower().nunique()
# census tail = misspelling suspects

# STEP 8 -- DUPLICATES
df.duplicated().sum(); df.duplicated(subset=[...]).sum()

# STEP 9 -- DOMAIN RULES & OUTLIERS
# rule masks: (df["col"] <= 0).sum() etc.       # impossible values
# boxplots / histograms / quantiles             # outliers (investigate, don't delete)

# STEP 10 -- WRITE IT DOWN: the data quality report
```

**The output of inspection is a *data quality report***: a written list of findings, each with (a) the issue, (b) its size (count / %), (c) the evidence (which check found it), and (d) an open question or recommendation. You'll write one in the capstone.

## 14. Bridge to the next lesson: issue → future fix

You are *not* fixing anything today — but here's the map of where each discovery leads, so the toolkit feels complete:

| Issue found | Discovered by | Fixed later with |
|---|---|---|
| Explicit missing values | `info()`, `isna()` | `dropna()` / `fillna()` (lesson 1!) |
| Disguised missing (`"N/A"`, sentinels) | censuses, histograms | `replace(...)`, `read_csv(na_values=...)` |
| Numbers as text (`"$4.00"`) | `info()`, coercion diagnostic | `str.replace` + `pd.to_numeric` |
| Mixed/ambiguous dates | `to_datetime` diagnostic | `pd.to_datetime` with explicit formats |
| Casing/whitespace variants | census, nunique comparison | `str.strip()`, `str.lower()` |
| Misspellings | census tail | `replace({"Cofee": "Coffee"})` |
| Exact duplicates | `duplicated()` | `drop_duplicates()` |
| Impossible values | domain-rule masks | case-by-case: fix, drop, or reinterpret |
| Junk header lines | raw-file peek | `read_csv(skiprows=...)` |

---

## 15. Capstone: the supplier file 🎯

The purchasing manager hands you `supplier_deliveries_raw.csv` — March deliveries from the chain's three suppliers — and asks the innocent question: *"What did we spend per supplier?"*

You know better than to answer immediately. **Your task: run the full inspection protocol and produce a data quality report.** No cleaning — diagnosis only.

**Step-by-step instructions:**

1. **Manual first** (Step 0). Peek at the raw lines. Something at the top will prevent a normal load — identify it and load the file correctly (hint from section 14's table: `skiprows`).
2. Run Steps 1–9 of the protocol. The file contains **at least seven distinct issue types** from this lesson. Find them all, and quantify each (how many rows / what %).
3. Watch the *column names* themselves (Step 3 — `df.columns` with `repr` or `list()`).
4. Write your **data quality report** in a markdown cell: one bullet per finding with issue, size, evidence, and — where relevant — an open question for the purchasing manager.
5. Finish with one sentence answering: *can "spend per supplier" be computed from this file as-is?*

In [ ]:
# Capstone workspace -- add as many cells as you need


In [ ]:
# (more capstone workspace)


*(Write your data quality report in this markdown cell — double-click to edit.)*

---

## 16. Solutions ✅

Solve everything yourself before reading on — inspection is a skill built by *looking*, not by reading answers.

### Exercises 4

In [ ]:
# 4.1 -- last 5 raw lines
with open("coffee_sales_raw.csv") as f:
    for line in f.readlines()[-5:]:
        print(repr(line))
# 4.2 -- examples visible in raw text: '$6.50'-style prices, 'DOWNTOWN ', 'downtown'

### Exercises 5

In [ ]:
# 5.1
print("Extra rows:", df.shape[0] - 400)
# 5.2 -- typical spots: N/A prices, $ prefixes, lowercase branches, rating 99, negative quantity
df.sample(10, random_state=7)

### Exercises 6

In [ ]:
# 6.1
print("Missing ratings:", df["customer_rating"].isna().sum())   # 408 - non-null count
# 6.2: with dtype str, .dt accessors fail and grouping by day/sorting chronologically is
# unreliable (string order != date order once formats are mixed), so no daily line chart.

### Exercises 7

In [ ]:
# 7.1
print("Distinct dates:", df["date"].nunique())
# March has 31 days -> more than 31 distinct values means at least one date is written
# in a second format (the same day spelled two ways), confirming a format problem.
# 7.2
print("quantity mean:", df["quantity"].mean().round(2),
      "| median:", df["quantity"].median())
# Close together -> no massive contamination (the few negatives barely move the mean),
# unlike customer_rating where the 99s tear mean and median apart.

### Exercises 8

In [ ]:
# 8.1
print(df["category"].value_counts(dropna=False))
# Finding A: 'Cofee' misspelling. Finding B: no NaN row appears -> no explicit missing.
# 8.2
n_na = (df["unit_price"] == "N/A").sum()
print(f"N/A prices: {n_na} = {n_na/len(df)*100:.1f}%")
# 8.3
print("Zero quantities:", (df["quantity"] == 0).sum())   # hypothesis tested: none

### Exercises 9

In [ ]:
# 9.1
conv_q = pd.to_numeric(df["quantity"], errors="coerce")
print("Unconvertible quantities:", (df["quantity"].notna() & conv_q.isna()).sum())
# 0 -> the column is numerically sound (its problem is impossible VALUES, not types)
# 9.2
print(df["date"].min())
# As text, '05/03/2026' < '2026-03-01' because '0' < '2' as characters --
# so the 'earliest date' is chosen by alphabet, not by calendar.

### Exercises 10

In [ ]:
# 10.1
print(df["product"].value_counts())
print("raw:", df["product"].nunique(),
      "normalized:", df["product"].str.strip().str.lower().nunique())
# 8 == 8 -> product is consistent. A clean verdict is a valid finding.
# 10.2
truly_downtown = df["branch"].str.strip().str.lower() == "downtown"
naive = df["branch"] == "Downtown"
print("Rows silently missed:", (truly_downtown & ~naive).sum())

### Exercises 11

In [ ]:
# 11.1
exact = df.duplicated().sum()
partial = df.duplicated(subset=["date", "branch", "product", "quantity"]).sum()
print("exact:", exact, "| partial:", partial)
# Partial is larger because fewer columns must match; two DIFFERENT sales can easily
# share date+branch+product+quantity, so partial matches are weak evidence.
# 11.2: because identical rows may be genuinely distinct events; deleting requires
# knowing whether the columns uniquely identify a transaction -- a domain question.

### Exercises 12

In [ ]:
# 12.1
price_num = pd.to_numeric(df["unit_price"], errors="coerce")
viol = price_num.notna() & ~price_num.between(0.50, 20)
print("Price rule violations:", viol.sum())
# 12.2
idx = df["quantity"].abs().idxmax()
print(df.loc[idx, ["product", "quantity"]])
# |quantity| max is 4 -> within the normal 1-4 range (sign aside): the extreme rows are
# IMPOSSIBLE (negative), not outliers -- there are no statistical outliers in quantity.

### Capstone — full worked solution

**Step 0/1 — manual peek and correct load:**

In [ ]:
with open("supplier_deliveries_raw.csv") as f:
    for line in f.readlines()[:6]:
        print(repr(line))

In [ ]:
# Two junk metadata lines before the header -> skip them
sup = pd.read_csv("supplier_deliveries_raw.csv", skiprows=2)
print(sup.shape)
print(list(sup.columns))   # note 'supplier ' and ' cost' -- whitespace in NAMES

In [ ]:
sup.info()

In [ ]:
sup.describe(include="all")

In [ ]:
# Missing & disguised
print(sup.isna().sum())
print()
print(sup["quantity_kg"].value_counts(dropna=False).head(8))

In [ ]:
# Type diagnostics
q = pd.to_numeric(sup["quantity_kg"], errors="coerce")
print("quantity_kg unconvertible:", (sup["quantity_kg"].notna() & q.isna()).sum(),
      "->", sup.loc[sup["quantity_kg"].notna() & q.isna(), "quantity_kg"].unique())
c = pd.to_numeric(sup[" cost"], errors="coerce")
print(" cost unconvertible:", (sup[" cost"].notna() & c.isna()).sum(),
      "-> sample:", sup.loc[sup[" cost"].notna() & c.isna(), " cost"].unique()[:4])
d = pd.to_datetime(sup["delivery_date"], format="%Y-%m-%d", errors="coerce")
print("dates off-format:", d.isna().sum(),
      "->", sup.loc[d.isna(), "delivery_date"].unique())

In [ ]:
# Categoricals
print(sup["supplier "].value_counts())
print("supplier raw:", sup["supplier "].nunique(),
      "normalized:", sup["supplier "].str.strip().str.lower().nunique())
print()
print(sup["item"].value_counts())

In [ ]:
# Duplicates, domain rules, outliers
print("exact duplicates:", sup.duplicated().sum())
print("negative quantities:", (q < 0).sum())
print("quantity quantiles:")
print(q.quantile([0.5, 0.95, 1.0]))   # max wildly beyond the typical range

**A model data quality report** (yours may be worded differently — the findings and quantification are what count):

> **Data quality report — supplier_deliveries_raw.csv (125 rows loaded)**
>
> 1. **Structural:** 2 metadata lines above the header (evidence: raw-file peek); loaded with `skiprows=2`. Column names contain whitespace: `'supplier '`, `' cost'` (evidence: `list(df.columns)`) — any code using clean names will hit `KeyError`.
> 2. **Duplicates:** 5 exact duplicate rows ≈ 4% (evidence: `duplicated().sum()`). `delivery_id` repeats for them → these are true re-records, not coincidences, since the ID should be unique.
> 3. **Disguised missing:** `quantity_kg` contains the string `"unknown"` (7 rows ≈ 6%) — the column loads as text because of it (evidence: `info()` + coercion diagnostic).
> 4. **Impossible values:** 4 negative quantities; and one delivery of 9999 kg against a median of ~41 kg and a 95th percentile under ~80 kg — either a typo or a unit error (evidence: quantile check). *Question for purchasing: can 9999 be real?*
> 5. **Numbers as text:** `cost` mixes `"1,234.56"` and `"$1,234.56"` — thousands separators plus a currency prefix (≈17% with `$`); wholly text-typed (evidence: coercion diagnostic).
> 6. **Mixed date formats:** 5 dates in `DD/MM/YYYY` among ISO dates (evidence: `to_datetime` format test). Ambiguous — *which convention does LogiTrack export?*
> 7. **Categorical variants:** `supplier` has 8+ raw labels collapsing to 3 after normalization (casing + trailing spaces); `item` contains the misspelling `"Arabca beans"` (3 rows) that normalization does **not** fix.
>
> **Verdict:** "spend per supplier" **cannot** be computed as-is: the supplier labels fragment the groups, cost is text, and duplicates would inflate totals by ~4%. All issues are fixable; none require discarding the file.

---

🎉 **Well done.** You now own the habit that quietly defines good analysts: **never trust a dataset you haven't interrogated.** Next lesson, we pick up the other half of the craft — actually fixing what you found.